# 02 · Parameter 与 Buffer

> **覆盖的类/函数**：`register_parameter`, `register_buffer`, `parameters`, `named_parameters`, `buffers`, `named_buffers`, `_named_members`, `get_parameter`, `get_buffer`  
> **PyTorch 源码**：[torch/nn/modules/module.py](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py)  
> **运行环境**：Python 3.9+, PyTorch 2.0+

---

## Part A: 源码阅读

In [1]:
import torch
import torch.nn as nn
import inspect

print(f'PyTorch 版本: {torch.__version__}')

PyTorch 版本: 2.8.0


### A.1 `register_parameter` — 注册可学习参数

`register_parameter` 将 `nn.Parameter` 对象存入 `_parameters` 字典。这是优化器追踪参数的入口。

In [2]:
print('register_parameter 源码:')
print(inspect.getsource(nn.Module.register_parameter))

register_parameter 源码:
    def register_parameter(self, name: str, param: Optional[Parameter]) -> None:
        r"""Add a parameter to the module.

        The parameter can be accessed as an attribute using given name.

        Args:
            name (str): name of the parameter. The parameter can be accessed
                from this module using the given name
            param (Parameter or None): parameter to be added to the module. If
                ``None``, then operations that run on parameters, such as :attr:`cuda`,
                are ignored. If ``None``, the parameter is **not** included in the
                module's :attr:`state_dict`.
        """
        if "_parameters" not in self.__dict__:
            raise AttributeError(
                "cannot assign parameter before Module.__init__() call"
            )

        elif not isinstance(name, str):
            raise TypeError(
                f"parameter name should be a string. Got {torch.typename(name)}"
         

**关键逻辑**：
- `value` 必须是 `None` 或 `nn.Parameter`（`torch.Tensor` 不行）
- `value is not None` → 存入 `_parameters[name]`；`value is None` → 删除已有参数
- 存入前检查是否有同名 Module 或 Buffer 冲突

### A.2 `register_buffer` — 注册不参与训练的持久化状态

Buffer 是不可学习的 Tensor，但会参与 `to(device)`、`state_dict()` 等操作。典型例子：BatchNorm 的 `running_mean`。

In [3]:
print('register_buffer 源码:')
print(inspect.getsource(nn.Module.register_buffer))

register_buffer 源码:
    def register_buffer(
        self, name: str, tensor: Optional[Tensor], persistent: bool = True
    ) -> None:
        r"""Add a buffer to the module.

        This is typically used to register a buffer that should not be
        considered a model parameter. For example, BatchNorm's ``running_mean``
        is not a parameter, but is part of the module's state. Buffers, by
        default, are persistent and will be saved alongside parameters. This
        behavior can be changed by setting :attr:`persistent` to ``False``. The
        only difference between a persistent buffer and a non-persistent buffer
        is that the latter will not be a part of this module's
        :attr:`state_dict`.

        Buffers can be accessed as attributes using given names.

        Args:
            name (str): name of the buffer. The buffer can be accessed
                from this module using the given name
            tensor (Tensor or None): buffer to be registered. If

**`persistent` 参数**：
- `True`（默认）：参与 `state_dict()`、`load_state_dict()`
- `False`：不参与序列化，但仍然参与 `to(device)` 和 `.cuda()`。适用于临时状态（如 RNN 的隐藏状态）

### A.3 `parameters()` / `named_parameters()` — 参数迭代

`named_parameters()` 返回 `(完整路径名, Parameter)` 对，`parameters()` 只返回 Parameter。两者都支持 `recurse=True/False`。

In [4]:
print('named_parameters 源码（精简）:')
print(inspect.getsource(nn.Module.named_parameters)[:600])

named_parameters 源码（精简）:
    def named_parameters(
        self, prefix: str = "", recurse: bool = True, remove_duplicate: bool = True
    ) -> Iterator[tuple[str, Parameter]]:
        r"""Return an iterator over module parameters, yielding both the name of the parameter as well as the parameter itself.

        Args:
            prefix (str): prefix to prepend to all parameter names.
            recurse (bool): if True, then yields parameters of this module
                and all submodules. Otherwise, yields only parameters that
                are direct members of this module.
            remove_duplicate (bool, 


### A.4 `_named_members` — `named_parameters` 和 `named_buffers` 的共享引擎

这是理解 Module 遍历机制的关键函数。它先通过 `named_modules()` 获取 Module 树，再对每个 Module 调用 `get_members_fn` 提取成员。

In [5]:
print('_named_members 源码:')
print(inspect.getsource(nn.Module._named_members))

_named_members 源码:
    def _named_members(
        self, get_members_fn, prefix="", recurse=True, remove_duplicate: bool = True
    ):
        r"""Help yield various names + members of modules."""
        memo = set()
        modules = (
            self.named_modules(prefix=prefix, remove_duplicate=remove_duplicate)
            if recurse
            else [(prefix, self)]
        )
        for module_prefix, module in modules:
            members = get_members_fn(module)
            for k, v in members:
                if v is None or v in memo:
                    continue
                if remove_duplicate:
                    memo.add(v)
                name = module_prefix + ("." if module_prefix else "") + k
                yield name, v



**三个关键参数**：
- `get_members_fn`：提取函数，`named_parameters` 传入 `lambda m: m._parameters.items()`
- `prefix`：名称前缀（递归时拼接 `layer.0.`）
- `recurse`：是否递归遍历子模块（`True` 默认 vs `False` 只取当前层的）

**去重机制**：`memo` set 追踪已 yield 的 Parameter id，防止共享参数被重复返回。

### A.5 `get_parameter` / `get_buffer` — 按点分隔路径查找

类似于 `get_submodule`，但返回的是 Parameter 或 Buffer 而非 Module。

In [6]:
print('get_parameter 源码:')
print(inspect.getsource(nn.Module.get_parameter))
print('\nget_buffer 源码:')
print(inspect.getsource(nn.Module.get_buffer))

get_parameter 源码:
    def get_parameter(self, target: str) -> "Parameter":
        """Return the parameter given by ``target`` if it exists, otherwise throw an error.

        See the docstring for ``get_submodule`` for a more detailed
        explanation of this method's functionality as well as how to
        correctly specify ``target``.

        Args:
            target: The fully-qualified string name of the Parameter
                to look for. (See ``get_submodule`` for how to specify a
                fully-qualified string.)

        Returns:
            torch.nn.Parameter: The Parameter referenced by ``target``

        Raises:
            AttributeError: If the target string references an invalid
                path or resolves to something that is not an
                ``nn.Parameter``
        """
        module_path, _, param_name = target.rpartition(".")

        mod: torch.nn.Module = self.get_submodule(module_path)

        if not hasattr(mod, param_name):
          

---

## Part B: 机制分析

### B.1 Parameter vs 普通 Tensor：完整行为差异表

| 行为 | `nn.Parameter`（注册为 parameter） | 普通 `torch.Tensor`（注册为 buffer） | 普通 `torch.Tensor`（直接赋值 `self.x=`） |
|------|--------------------------------------|---------------------------------------|--------------------------------------------|
| 存储位置 | `_parameters` | `_buffers` | `__dict__`（普通属性） |
| 被优化器更新 | ✅ | ❌ | ❌ |
| `model.to(device)` 移动 | ✅ | ✅ | ❌ |
| `state_dict()` 保存 | ✅ | ✅（persistent=True） | ❌ |
| `model.float()/.half()` 转换 | ✅ | ✅ | ❌ |
| `requires_grad` 默认 | `True` | `False` | `False` |
| `named_parameters()` 可见 | ✅ | ❌ | ❌ |
| `named_buffers()` 可见 | ❌ | ✅ | ❌ |

> ⚠️ **核心规则**：想让 Tensor 被 PyTorch Module 系统管理（设备迁移、序列化），必须注册为 parameter 或 buffer。

### B.2 `_named_members` 的去重逻辑

当一个 Parameter 对象被多个 Module 共享时（如 weight tying），`memo` set 确保它只在第一次遇到时被 yield。

```
for module_prefix, module in named_modules():  # DFS
    for k, v in get_members_fn(module):
        if v is None or id(v) in memo:
            continue  # 跳过已见过的
        memo.add(id(v))
        yield f'{module_prefix}.{k}', v
```

---

## Part C: 动手实验

In [7]:
# ===== 实验 1：Parameter vs Tensor vs Buffer 的行为对比 =====
print('=== 实验 1：三种注册方式的行为对比 ===\n')

class CompareModule(nn.Module):
    def __init__(self):
        super().__init__()
        # 方式 1：注册为 Parameter（推荐用于权重）
        self.register_parameter('weight', nn.Parameter(torch.randn(3, 3)))
        # 方式 2：注册为 Buffer（推荐用于非学习状态）
        self.register_buffer('running_mean', torch.zeros(3))
        # 方式 3：直接赋值普通属性（⚠️ 不推荐用于 Tensor）
        self.counter = torch.tensor([0.0])  # 这种会被视为普通 Python 属性

model = CompareModule()
print(f'_parameters:   {list(model._parameters.keys())}')
print(f'_buffers:      {list(model._buffers.keys())}')
print(f'__dict__ keys: {[k for k in model.__dict__ if not k.startswith("_")]}')

print(f'\nweight  类型: {type(model.weight).__name__}')
print(f'weight  requires_grad: {model.weight.requires_grad}')
print(f'running_mean requires_grad: {model.running_mean.requires_grad}')
print(f'counter  requires_grad: {model.counter.requires_grad}')

=== 实验 1：三种注册方式的行为对比 ===

_parameters:   ['weight']
_buffers:      ['running_mean']
__dict__ keys: ['training', 'counter']

weight  类型: Parameter
weight  requires_grad: True
running_mean requires_grad: False
counter  requires_grad: False


In [8]:
# 验证 state_dict() 包含哪些
sd = model.state_dict()
print('state_dict() 中的 key:')
for k in sd:
    print(f'  {k}: shape={sd[k].shape}')
print(f'\n⚠️ 注意: counter（普通属性）不在 state_dict 中！')

state_dict() 中的 key:
  weight: shape=torch.Size([3, 3])
  running_mean: shape=torch.Size([3])

⚠️ 注意: counter（普通属性）不在 state_dict 中！


In [9]:
# 验证 to(device) 行为
if torch.cuda.is_available():
    model_cuda = CompareModule().cuda()
    print('cuda 后:')
    print(f'  weight device: {model_cuda.weight.device}')
    print(f'  running_mean device: {model_cuda.running_mean.device}')
    print(f'  counter device: {model_cuda.counter.device}  ← 普通 Tensor 还在 CPU！')
else:
    print('无 GPU，跳过 cuda 测试。可以想象 counter 不会被迁移到 GPU。')

无 GPU，跳过 cuda 测试。可以想象 counter 不会被迁移到 GPU。


In [10]:
# ===== 实验 2：persistent=False Buffer =====
print('=== 实验 2：persistent=False Buffer ===\n')

class PersistentModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('persistent_buf', torch.tensor([1.0, 2.0]), persistent=True)
        self.register_buffer('temp_buf', torch.tensor([3.0, 4.0]), persistent=False)

pm = PersistentModule()
print('state_dict() keys:')
for k in pm.state_dict():
    print(f'  {k}: {pm.state_dict()[k]}')

print(f'\n_named_buffers 中的 key:')
for name, buf in pm.named_buffers():
    print(f'  {name}: {buf}')

print(f'\n✅ persistent=False 的 Buffer:')
print(f'  - 不出现在 state_dict() 中（不会被 torch.save 保存）')
print(f'  - 仍然出现在 named_buffers() 中')
print(f'  - 仍然参与 to(device) 迁移')

=== 实验 2：persistent=False Buffer ===

state_dict() keys:
  persistent_buf: tensor([1., 2.])

_named_buffers 中的 key:
  persistent_buf: tensor([1., 2.])
  temp_buf: tensor([3., 4.])

✅ persistent=False 的 Buffer:
  - 不出现在 state_dict() 中（不会被 torch.save 保存）
  - 仍然出现在 named_buffers() 中
  - 仍然参与 to(device) 迁移


In [11]:
# ===== 实验 3：共享参数去重 =====
print('=== 实验 3：共享 Parameter 的去重行为 ===\n')

# 场景：weight tying — encoder 和 decoder 共享同一个 weight
shared_weight = nn.Parameter(torch.randn(10, 10))

class SharedModule(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.encoder = nn.Linear(10, 10)
        self.encoder.weight = weight  # 替换为共享 weight
        self.decoder = nn.Linear(10, 10)
        self.decoder.weight = weight  # 同一个 weight！

model = SharedModule(shared_weight)

print('named_parameters():')
for name, param in model.named_parameters():
    print(f'  {name}: id={id(param)} shape={param.shape}')

print(f'\n✅ encoder.weight 和 decoder.weight 指向同一个对象（id 相同）')
print(f'  但 named_parameters() 只返回一次（去重）')
print(f'  参数总数: {sum(1 for _ in model.parameters())}（有 2 个 Linear，但共享 1 个 weight）')

=== 实验 3：共享 Parameter 的去重行为 ===

named_parameters():
  encoder.weight: id=4605039296 shape=torch.Size([10, 10])
  encoder.bias: id=4605039616 shape=torch.Size([10])
  decoder.bias: id=4605039456 shape=torch.Size([10])

✅ encoder.weight 和 decoder.weight 指向同一个对象（id 相同）
  但 named_parameters() 只返回一次（去重）
  参数总数: 3（有 2 个 Linear，但共享 1 个 weight）


In [12]:
# ===== 实验 4：named_parameters 的 prefix 和 recurse =====
print('=== 实验 4：prefix 命名规则与 recurse 参数 ===\n')

class NestedModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 3)       # weight + bias
        self.nest = nn.Sequential(
            nn.Linear(3, 2),              # weight + bias
            nn.ReLU()
        )

model = NestedModule()

print('recurse=True（默认）:')
for name, param in model.named_parameters(recurse=True):
    print(f'  {name:30s} shape={param.shape}')

print('\nrecurse=False（仅当前层）:')
for name, param in model.named_parameters(recurse=False):
    print(f'  {name:30s} shape={param.shape}')

print('\n带 prefix:')
for name, param in model.named_parameters(prefix='model'):
    print(f'  {name:30s} shape={param.shape}')

=== 实验 4：prefix 命名规则与 recurse 参数 ===

recurse=True（默认）:
  fc1.weight                     shape=torch.Size([3, 5])
  fc1.bias                       shape=torch.Size([3])
  nest.0.weight                  shape=torch.Size([2, 3])
  nest.0.bias                    shape=torch.Size([2])

recurse=False（仅当前层）:

带 prefix:
  model.fc1.weight               shape=torch.Size([3, 5])
  model.fc1.bias                 shape=torch.Size([3])
  model.nest.0.weight            shape=torch.Size([2, 3])
  model.nest.0.bias              shape=torch.Size([2])


In [13]:
# ===== 实验 5：get_parameter / get_buffer 点分隔路径查找 =====
print('=== 实验 5：get_parameter / get_buffer 路径查找 ===\n')

model = NestedModule()

# 用点分隔路径直接获取深层参数
param = model.get_parameter('nest.0.weight')  # Sequential 的第 0 个是 Linear
print(f"get_parameter('nest.0.weight'): shape={param.shape}")

# 路径不存在时
try:
    model.get_parameter('nonexistent')
except AttributeError as e:
    print(f"get_parameter('nonexistent'): {e}")

# get_buffer 同理
model.register_buffer('my_buf', torch.zeros(3))
buf = model.get_buffer('my_buf')
print(f"get_buffer('my_buf'): {buf}")

print(f'\n✅ get_parameter/get_buffer 底层使用 get_submodule 做路径解析')

=== 实验 5：get_parameter / get_buffer 路径查找 ===

get_parameter('nest.0.weight'): shape=torch.Size([2, 3])
get_parameter('nonexistent'): NestedModule has no attribute `nonexistent`
get_buffer('my_buf'): tensor([0., 0., 0.])

✅ get_parameter/get_buffer 底层使用 get_submodule 做路径解析


---

## Part D: 小结

### 要点清单

- [x] `register_parameter(name, param)` 将 `nn.Parameter` 存入 `_parameters`（value 必须为 `None` 或 `Parameter`）
- [x] `register_buffer(name, tensor, persistent=True)` 将 Tensor 存入 `_buffers`
- [x] `persistent=False` 的 Buffer 不参与 `state_dict()` 序列化，但仍参与设备迁移
- [x] `named_parameters()` / `named_buffers()` 共用 `_named_members` 引擎，支持 `recurse` 和 `prefix`
- [x] `_named_members` 用 `memo` set 做去重（基于 `id(param)`），防止共享参数重复出现
- [x] `get_parameter(target)` / `get_buffer(target)` 支持点分隔路径查找深度嵌套参数
- [x] 普通 `self.x = tensor` 不会注册：不参与设备迁移、不会被 `state_dict` 保存、不被优化器更新

### 与后续 Notebook 的关联

- **Notebook 03** 将深入讲解 `named_modules()` 的 DFS 遍历机制（`_named_members` 的底层引擎）
- **Notebook 04** 将揭示 `_apply` 如何递归处理 `_parameters` 和 `_buffers` 实现设备/类型转换
- **Notebook 07** 将详细展开 `state_dict()` / `load_state_dict()` 的序列化流程

### 🏠 导航

← [返回首页流程图](../../index.html) | [下一章：03 · Module 层级导航](03_module_hierarchy.html)